**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**05 | Participant-Balanced Descriptive Statistics**

![Source-aligned participant-level gaze statistics and exploratory bootstrap design](../../images/05_participant_balanced_statistics.svg)

**Objective:** Describe GazeBase gaze-feature distributions by task while keeping the **participant**, rather than the gaze sample, event or recording, as the independent resampling unit.

**Input evidence:** Notebook 01 recorded 10 participants and 40 task recordings. Notebook 04 has now executed and saved **40 per-record feature rows for 10 participants**, including I-VT event counts, fixation duration, centroid path length, speed and uncalibrated pupil summaries; its saved missingness display reported **no unavailable feature values**. These are upstream observations, not Notebook 05 statistics.

**Scientific design:** Match the Notebook 04 feature table to Notebook 01 source quality by exact record identity and validity fraction; aggregate records within participant and task; calculate descriptive means, sample standard deviations and exploratory percentile bootstrap intervals from participant values (**500 draws, seed 17**). No hypothesis testing, causal conclusion, population generalisation or independent validation is claimed.

**Execution status:** Notebook 05 has not been run on these original-data outputs. The introductory SVG shows observed upstream scope together with labelled method schematics; its uncertainty intervals are not empirical results until this Notebook is executed.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_statistics import (
    validate_record_quality_alignment, participant_summary,
    summarise_participant_metrics,
)

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover and Reconcile Notebook 01 and Notebook 04 Tables**

Read the compact `01_quality_by_record.csv` and `04_features_by_record.csv` outputs, without loading multi-million-row gaze tables. Both contain `record_id`, `participant_id`, `task` and `valid_fraction`.

Check one-to-one record identity, matched task and participant codes, and agreement of original versus processed valid fractions. This verifies that the descriptive input still represents the same source subset. A mismatch is reported explicitly rather than silently dropping records or treating missing data as zero. Display a short reconciliation preview and the number of independent people before proceeding.


In [ ]:
# =============== 02. Recover Feature and Source Quality Tables ===============

# ----------------------- 1. Verify Notebook Outputs --------------------------
ensure_stage_inputs(output_dir, 1, ("01_quality_by_record.csv",))
ensure_stage_inputs(output_dir, 4, ("04_features_by_record.csv",))
quality = read_stage(
    output_dir / "01_quality_by_record.csv",
    ("record_id", "participant_id", "task", "valid_fraction"),
)
features = read_stage(
    output_dir / "04_features_by_record.csv",
    ("record_id", "participant_id", "task", "valid_fraction"),
)

# ---------------- 2. Reconcile Exact Source Record Identity ------------------
quality_alignment = validate_record_quality_alignment(features, quality)
print("Matching source records:", len(quality_alignment))
print("Independent participants:", features["participant_id"].nunique())
print("Tasks:", ", ".join(sorted(features["task"].unique())))
print("Maximum validity discrepancy:",
      quality_alignment["absolute_valid_fraction_difference"].max())
display(quality_alignment[[
    "record_id", "participant_id_notebook04", "task_notebook04",
    "valid_fraction_notebook04", "valid_fraction_notebook01",
]].head(8))


**03. Aggregate Within Participants and Tasks**

Repeated observations within a person are not independent participants. Group the validated record features first by `participant_id` and `task`, averaging within those groups where multiple source recordings exist. In the current bounded Round 1 / Session 1 subset, each of the ten selected people contributes one FXS, HSS, RAN and TEX recording.

The four prespecified descriptive metrics are valid gaze fraction, mean detected fixation duration, approximate fixation-centroid path length and mean I-VT saccade peak speed. They are computed from provisional detector outputs; the extremely high measured velocities noted in Notebook 02 remain a potential source of distortion, not observations to remove automatically. Retain the number of participants contributing to each feature.


In [ ]:
# ===================== 03. Aggregate Within Participants =====================

# ---------------------- 1. Prespecified Source Metrics -----------------------
METRICS = (
    "valid_fraction", "mean_fixation_ms", "scanpath_length_dva",
    "mean_saccade_peak_dva_s",
)
participant = participant_summary(features, METRICS)
participant.to_csv(output_dir / "05_participant_features.csv", index=False)

# -------------------- 2. Show Independent Analysis Units ---------------------
print("Participant-by-task rows:", len(participant))
print("Unique participants:", participant["participant_id"].nunique())
display(participant.head(12))


**04. Calculate Exploratory Participant-Level Uncertainty**

For each task and available metric, summarise the participant-level mean and sample standard deviation. Construct an exploratory percentile interval by drawing **500 bootstrap samples of participants with replacement**, using fixed random seed **17**. Every resample consists of independent participant values within a task, **not** individual gaze samples or events.

The resulting percentile interval is descriptive and may be unstable with only ten individuals. Rows with fewer than two contributing participants have undefined intervals and remain missing rather than receiving fabricated bounds. These are not confirmatory confidence guarantees, formal between-task tests, or claims about unseen populations.


In [ ]:
# ============ 04. Calculate Participant-Level Uncertainty ================

# --------------------- 1. Fixed Exploratory Bootstrap ------------------------
BOOTSTRAP_DRAWS = 500
BOOTSTRAP_SEED = 17
stats = summarise_participant_metrics(
    participant, METRICS, draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED
)
stats.to_csv(output_dir / "05_descriptive_statistics.csv", index=False)

# --------------------- 2. Check Supported Feature Rows -----------------------
print("Descriptive task-feature rows:", len(stats))
print("Unique source tasks:", stats["task"].nunique())
display(stats[[
    "task", "feature", "n_independent_participants", "participant_mean",
    "participant_sd", "bootstrap_ci_lower", "bootstrap_ci_upper",
    "interval_status",
]])


**05. Plot Participant-Weighted Descriptive Findings**

Visualise the participant-level mean of `valid_fraction` for each of the four gaze tasks, with the corresponding **exploratory 2.5th–97.5th percentile bootstrap interval** and the number of independent contributors. This interval comes from the participant-level resampling described above; the chart is not a repeated-sample CI.

Task comparisons are descriptive only. Differences may reflect stimulus type, calibration and tracking conditions, missing gaze and participant heterogeneity. No statistical significance, causal influence or cognitive outcome is inferred.


In [ ]:
# ============ 05. Plot Participant-Weighted Descriptive Findings =============

# ---------------- 1. Select Supported Gaze-Quality Estimates -----------------
valid = (
    stats.loc[stats["feature"].eq("valid_fraction")]
    .sort_values("task").reset_index(drop=True)
)
if valid.empty:
    raise ValueError("No participant-level valid-fraction summary to plot.")

# ------------------ 2. Draw Means and Bootstrap Intervals -------------------
fig, ax = plt.subplots(figsize=(8.5, 4.5))
positions = np.arange(len(valid))
ax.scatter(
    positions, valid["participant_mean"], marker="o", s=45,
    label="Participant mean", zorder=3,
)
for i, row in valid.iterrows():
    lo, hi = row["bootstrap_ci_lower"], row["bootstrap_ci_upper"]
    if np.isfinite(lo) and np.isfinite(hi):
        ax.vlines(i, lo, hi, lw=2.1)
        ax.hlines([lo, hi], i - 0.09, i + 0.09, lw=1.6)
    ax.annotate(
        f"n={int(row['n_independent_participants'])}",
        (i, float(row["participant_mean"])),
        xytext=(0, 9), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_xticks(positions, valid["task"])
ax.set(
    ylabel="Participant-level valid gaze fraction",
    xlabel="Gaze task",
    ylim=(0, 1.03),
    title="Exploratory Gaze Quality by Task",
)
ax.set_xlim(-0.55, len(valid) - 0.45)
ax.grid(axis="y", alpha=0.2)
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, output_dir / "05_statistics.png")
plt.show()


**06. Scientific Interpretation and Output Download**

Notebook 05 produces `05_participant_features.csv` (participant-by-task summaries), `05_descriptive_statistics.csv` (feature-specific participant support, descriptive moments and bootstrap limits), and `05_statistics.png` (task-wise valid gaze fraction with exploratory intervals).

Interpret these statistics relative to the **ten independent participants** in this bounded selection, the fixed task protocol, the I-VT detector's provisional thresholds and unresolved extreme gaze speeds. Do not treat repeated samples as independent replicates, infer reading comprehension or apply a confirmatory significance interpretation. The output ZIP supports reuse in Notebook 07 without adding generated tables to GitHub.


In [ ]:
# ================ 06. Scientific Interpretation and Download ===============

# -------------------- 1. Confirm Current Scientific Outputs ------------------
print("Participant summaries:", output_dir / "05_participant_features.csv")
print("Task statistics:", output_dir / "05_descriptive_statistics.csv")
print("Uncertainty figure:", output_dir / "05_statistics.png")
print("Independent unit: participant; intervals are exploratory.")

# ------------------------ 2. Download Notebook Outputs -----------------------
download_outputs_from_colab("eye_tracking_foundations/output", "05")
